# NOMAD: Automated Protein Isoform Deconvolution

This notebook demonstrates the recommended workflow for analyzing precursor-level mass spectrometry data using NOMAD. We will perform graph-based deconvolution to resolve isoform-specific signals and then map their pharmacological responses.

In [1]:
%load_ext autoreload
%autoreload 2

import polars as pl
from nomad import Nomad

# 1. Setup Input Paths
fasta_path = "/home/j.angelis/decryptE/homo_sapiens.fasta"
evidence_path = "/cmnfs/home/j.angelis/Projects/NOMAD/nomad/experiment/combined_ion.tsv"
metadata_path = "experiment/data/dose_metadata.csv"

# 2. Initialize the NOMAD Orchestrator
nm = Nomad(num_workers=12, metadata=metadata_path)

## 1. Quantification & NMF Deconvolution

The `quantify` method executes the entire structural digestion and signal factorization pipeline. It returns a performance dictionary containing Cross-Validation (CV) metrics including Spearman correlations and replicate-level R² values.

In [2]:
# Run the full quantification pipeline
stats = nm.quantify(fasta_path, evidence_path)

print("\n[*] Deconvolution Performance:")
for metric, value in stats.items():
    print(f"  > {metric:20}: {value:.4f}")
nm.plot_performance().show()

 [*] Parsing FASTA file: 468464it [00:00, 1318495.78it/s]


 [*] Parsing FASTA file complete.
 [*] Digesting 42367 proteins using 12 workers...


 [*] Digesting batches: 100%|██████████| 25/25 [00:07<00:00,  3.25it/s]


 [*] Adding 2863510 peptides and 9493086 edges...
 [*] Digestion complete.
 [*] Normalizing intensities to global target: 1.14e+07
 [*] Found 3040030 observations.
 [*] Filtered down to 3025243 observations matching digestion (retaining only those seen in >=3 samples).
 [*] Quantification parsing complete.


/cmnfs/home/j.angelis/Projects/NOMAD/nomad/src/nomad/utils/nmf/averaging.py:44: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at ../aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  res[drug] = torch.sparse_csr_tensor(
/cmnfs/home/j.angelis/Projects/NOMAD/nomad/src/nomad/utils/nmf/averaging.py:44: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at ../aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  res[drug] = torch.sparse_csr_tensor(
/cmnfs/home/j.angelis/Projects/NOMAD/nomad/src/nomad/utils/nmf/averaging.py:44: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to http

 [*] Fitting on GPUs:   0%|          | 0/7780 [00:00<?, ?it/s]

/cmnfs/home/j.angelis/Projects/NOMAD/nomad/src/nomad/utils/nmf/averaging.py:44: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at ../aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  res[drug] = torch.sparse_csr_tensor(



[*] Deconvolution Performance:
  > pearson_raw         : 0.7230
  > pearson_log10       : 0.6993
  > pearson_sqrt        : 0.3727
  > spearman_rho        : 0.8927
  > rep_pearson_raw     : 0.8199
  > rep_pearson_log10   : 0.7570
  > rep_pearson_sqrt    : 0.8876
  > rep_spearman_rho    : 0.9195
  > rs_pearson_raw      : 0.9896
  > rs_pearson_log10    : 0.9600
  > rs_pearson_sqrt     : 0.9758
  > rs_spearman_rho     : 0.9604


## 2. Pharmacological Significance Testing

Once isoforms are deconvolved, we fit their intensity profiles across drug concentrations using a 4-parameter log-logistic model. This step identifies which protein isoforms show significant response trends.

In [3]:
# Perform Dose-Response analysis
results_df = nm.dose_response_analyze(alpha=0.05, fc_lim=1.0)

print(f"\n[*] Summary of Findings:")
print(f"  > Total Deconvolved Isoforms: {len(results_df)}")
print(f"  > Statistically Significant:  {len(results_df.filter(pl.col('significant_trend')))}")

AttributeError: 'Nomad' object has no attribute 'dose_response_analyze'

## 3. Interactive Visualization

NOMAD provides several interactive tools to validate the deconvolution results.

In [ ]:
# 3.A Volcano Plot: Global view of isoform responses
nm.volcano_plot()

AttributeError: 'Nomad' object has no attribute 'volcano_plot'

In [ ]:
# 3.B Dose-Response Curves: Deep dive into a specific protein
target_protein = "Q9NYL2-3"
nm.dose_response_comparison(target_protein)

AttributeError: 'Nomad' object has no attribute 'dose_response_comparison'

In [7]:
# 3.C Structural Evidence: Precursor-to-Isoform mapping (H-matrix)
nm.structural_evidence("Q16526")

## 4. Final Export & Pathway Exploration

We can export the results for external use and build a portable Pathway Explorer web application.

In [ ]:
# Export results to CSV
nm.export_results("nomad_final_results.csv")

# Build the standalone Reactome Pathway Explorer App
nm.build_pathway_explorer(out_dir="pathway_explorer_app", forced_pathways=["map04110", "map05200"])

AttributeError: 'Nomad' object has no attribute 'export_results'

In [ ]:
#import pickle
#with open("nm_obj.pkl", "wb") as file:
#    pickle.dump(nm, file)

In [ ]:
#import pickle
#with open("nm_obj.pkl", "rb") as file:
#    nm = pickle.load(file)

/cmnfs/home/j.angelis/Projects/NOMAD/nomad/.venv/lib/python3.12/site-packages/torch/_utils.py:315: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at ../aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  result = torch.sparse_compressed_tensor(
